# S01 · From Molecules to Typed Graphs: Fundamentals

<div class="alert alert-block alert-info">
<b>Welcome to SynEdu.</b><br>
This talktorial is part of <b>SynEdu</b>, a lightweight, research-oriented teaching series built around the
<b>Syn</b> ecosystem and <b>RDKit</b> for practical, reproducible chemical and reaction modeling.
</div>

<div class="alert alert-block alert-success">
<b>What you will gain.</b><br>
You will learn how to represent molecules as <b>typed graphs</b> and how to reason about
structure-preserving matches that underpin graph-based reaction modeling: <b>graph morphisms</b>,
<b>isomorphism</b>, and <b>automorphisms</b>.
</div>

<div class="alert alert-block alert-warning">
<b>Reproducibility note.</b><br>
This notebook is self-contained and meant to be run top-to-bottom. If you are following the full SynEdu series,
continue with <b>S02</b> for <i>subgraph isomorphism</i> and <i>MCS</i>.
</div>

---

## Roadmap (S01)

0. **Setup & data**
1. **Theory: typed graphs and morphisms**
2. **RDKit → NetworkX typed graphs**
3. **Round-trip: NetworkX → RDKit and sanity checks**
4. **Isomorphism, automorphisms, and orbits**


## 0. Setup & data

In [ ]:
import rdkit
from rdkit import Chem
from rdkit.Chem import rdFMCS
import networkx as nx
import pandas as pd
from pathlib import Path

print("RDKit version:", rdkit.__version__)
print("NetworkX version:", nx.__version__)


RDKit version: 2025.09.3
NetworkX version: 3.6.1


In [ ]:
DATA_DIR = Path("data")
CSV_PATH = DATA_DIR / "molecules.csv"
df = pd.read_csv(CSV_PATH)
display(df)

,smiles,name
0,CCO,ethanol
1,CCCl,chloroethane
2,CCOCC,diethyl ether
3,O,water
4,C(C(=O)O)N,glycine


## 1. Theory: Typed Graphs and Morphisms

### 1.1 Typed molecular graphs

In computational reaction modeling, we represent molecules as **typed graphs** so that “matching” respects
chemical identity (elements, charges, bond orders), not just connectivity.

A **typed graph** is a quadruple

$$
G = (V, E, \tau_V, \tau_E),
$$

where:

- **Vertices** $V$ represent **atoms**.
- **Edges** $E \subseteq \{\{u,v\}\mid u,v\in V,\ u\neq v\}$ represent **bonds** (finite, undirected, simple: no loops, no parallel edges).
- $\tau_V: V \to \mathcal{A}_V$ assigns **atom attributes** (chemical labels).
- $\tau_E: E \to \mathcal{A}_E$ assigns **bond attributes** (chemical labels).

We often write $V(G)$ and $E(G)$ for the vertex and edge sets of $G$. For a vertex $v\in V(G)$:

- neighbourhood:
  $$
  N_G(v)=\{w\in V(G)\mid vw\in E(G)\},
  $$
- degree:
  $$
  \deg_G(v)=|N_G(v)|.
  $$

#### Labels (typed graphs)

“Types” are encoded as labelling maps

$$
\ell_V: V(G)\to L_V,\qquad \ell_E: E(G)\to L_E,
$$

where $L_V$ and $L_E$ are finite, non-empty label sets.
For molecular graphs we use the chemistry-specific notation:

$$
a_G: V(G)\to L_V \quad\text{(atom labels)},\qquad
b_G: E(G)\to L_E \quad\text{(bond labels)}.
$$

Let $\mathcal{G}$ denote the class of all labelled molecular graphs equipped with $(a_G,b_G)$.
In chemistry, $a_G(v)$ encodes *what atom this is* (element, charge, aromaticity, …), 
while $b_G(uv)$ encodes *what bond this is* (order, aromaticity, ring status, …).

---

### 1.2 Graph morphisms

Let $G,H\in\mathcal{G}$ be labelled molecular graphs with atom- and bond-labelling functions
$(a_G,b_G)$ and $(a_H,b_H)$.

A **(labelled) graph morphism** from $G$ to $H$ is a map

$$
\varphi:V(G)\to V(H)
$$

that preserves **chemical identity at atoms**, and preserves **bonds and their types**.
Formally, $\varphi$ must satisfy:

#### (M1) Atom-label preservation
For every atom $v\in V(G)$,

$$
a_H(\varphi(v)) = a_G(v).
$$

> **Chemistry meaning**  
> $\varphi$ never maps a carbon to a nitrogen, or a neutral atom to a charged atom, if those are encoded in $a_\cdot$.

#### (M2) Adjacency (bond existence) preservation
For every bond $uv\in E(G)$,

$$
\varphi(u)\varphi(v)\in E(H).
$$

> **Chemistry meaning**  
> Bonded atoms in $G$ must map to bonded atoms in $H$ (no bond can “disappear” under the map).

#### (M3) Bond-label preservation
For every bond $uv\in E(G)$,

$$
b_H\!\big(\varphi(u)\varphi(v)\big) = b_G(uv).
$$

> **Chemistry meaning**  
> If $uv$ is a double bond in $G$, its image must be a double bond in $H$ (and likewise for aromaticity if included).

> **Summary**  
> A morphism $\varphi$ is a label-preserving embedding of the local chemical graph of $G$ into $H$.  
> It preserves “what atoms are” and “how they are connected”.

---

#### Compatibility predicates (practical generalization)

In practice, chemoinformatics representations may differ (e.g. aromatic vs Kekulé, resonance conventions).
We therefore sometimes relax strict equality into **compatibility**:

- atom compatibility:
  $$
  a_H(\varphi(v)) \sim_V a_G(v),
  $$
- bond compatibility:
  $$
  b_H(\varphi(u)\varphi(v)) \sim_E b_G(uv),
  $$

where $\sim_V$ and $\sim_E$ encode allowed correspondences (e.g. “aromatic bond” compatible with alternating single/double under a chosen model).

> **For chemists**  
> This is where you decide whether two representations should be considered “the same chemistry”.  
> For example, strict matching distinguishes aromatic vs Kekulé; compatibility matching can treat them as equivalent.

---

#### Standard special cases (matching tasks)

- **Monomorphism (injective morphism)**  
  $\varphi$ is injective (distinct atoms in $G$ map to distinct atoms in $H$).  
  $\rightarrow$ This is the formal object behind **substructure search / subgraph isomorphism**.

- **Isomorphism**  
  $\varphi$ is bijective and $\varphi^{-1}$ is also a morphism.  
  $\rightarrow$ We write $G\simeq H$ (same molecule up to relabeling).

- **Automorphism**  
  An isomorphism $\varphi:G\to G$.  
  $\rightarrow$ Encodes **molecular symmetry**, which can multiply equivalent matches and motivates deduplication.

> **Interpretation for reaction rules**  
> Applying a graph-rewrite rule begins by finding a **monomorphism** from the rule LHS into the host molecule.  
> Automorphisms (symmetry) can generate many equivalent embeddings; later SynEdu notebooks introduce symmetry-aware deduplication.

## 2. RDKit ⇄ NetworkX: Typed Molecular Graphs

In SynEdu, **RDKit** and **NetworkX** play complementary roles:

- **RDKit** is the chemical authority: sanitization, valence rules, aromaticity perception, and canonicalization.
- **NetworkX** provides an explicit, inspectable graph representation used for matching, symmetry analysis,
  and later graph rewriting.

To ensure that graph-based operations remain chemically meaningful, we require a **reversible interface**
between the two representations.

In [3]:
from typing import Dict
import networkx as nx
from rdkit import Chem
import rdkit

In [4]:
# RDKit -> NetworkX

def mol_to_graph(mol: Chem.Mol, include_implicit_h: bool = True) -> nx.Graph:
    """
    Convert RDKit Mol -> typed NetworkX graph.

    :param mol: RDKit Mol (assumed sanitized).
    :param include_implicit_h: If True, store total H count per atom as ``total_h``.
    :returns: networkx.Graph with atom/bond labels as node/edge attributes.
    """
    G = nx.Graph()

    for atom in mol.GetAtoms():
        i = atom.GetIdx()
        attrs: Dict[str, object] = {
            "symbol": atom.GetSymbol(),
            "formal_charge": int(atom.GetFormalCharge()),
            "aromatic": bool(atom.GetIsAromatic()),
            "chiral_tag": str(atom.GetChiralTag()),
        }
        if include_implicit_h:
            attrs["total_h"] = int(atom.GetTotalNumHs())
        G.add_node(i, **attrs)

    for bond in mol.GetBonds():
        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()
        order = int(round(bond.GetBondTypeAsDouble()))
        G.add_edge(
            u, v,
            order=order,
            aromatic=bool(bond.GetIsAromatic()),
            in_ring=bool(bond.IsInRing()),
        )

    G.graph["source"] = "rdkit"
    G.graph["rdkit_version"] = rdkit.__version__
    return G


In [5]:
# NetworkX to rdkit
def graph_to_mol(G: nx.Graph, make_explicit_h: bool = False) -> Chem.Mol:
    """
    Reconstruct RDKit Mol from typed NetworkX graph (inverse of ``mol_to_graph`` up to sanitization).

    :param G: Typed molecular graph produced by ``mol_to_graph``.
    :param make_explicit_h: If True and ``total_h`` exists, add explicit H atoms (best-effort).
    :returns: Sanitized RDKit Mol.
    """
    rw = Chem.RWMol()
    nx_to_rdk: Dict[int, int] = {}

    # atoms
    for node in sorted(G.nodes()):
        n = G.nodes[node]
        atom = Chem.Atom(n.get("symbol", "C"))
        atom.SetFormalCharge(int(n.get("formal_charge", 0)))
        if n.get("aromatic", False):
            atom.SetIsAromatic(True)

        ch_tag = n.get("chiral_tag")
        if ch_tag and ch_tag != "CHI_UNSPECIFIED":
            try:
                atom.SetChiralTag(getattr(Chem.rdchem.ChiralType, ch_tag))
            except Exception:
                pass  # best-effort only

        nx_to_rdk[node] = rw.AddAtom(atom)

    # bonds
    for u, v, e in G.edges(data=True):
        order = int(e.get("order", 1))
        btype = {
            1: Chem.rdchem.BondType.SINGLE,
            2: Chem.rdchem.BondType.DOUBLE,
            3: Chem.rdchem.BondType.TRIPLE,
        }.get(order, Chem.rdchem.BondType.SINGLE)
        rw.AddBond(nx_to_rdk[u], nx_to_rdk[v], btype)

    mol = rw.GetMol()

    # optional explicit H
    if make_explicit_h:
        for node, rdk_idx in nx_to_rdk.items():
            total_h = G.nodes[node].get("total_h")
            if total_h is None:
                continue
            atom = mol.GetAtomWithIdx(rdk_idx)
            current_h = sum(1 for n in atom.GetNeighbors() if n.GetSymbol() == "H")
            for _ in range(max(int(total_h) - current_h, 0)):
                h_idx = mol.AddAtom(Chem.Atom("H"))
                mol.AddBond(rdk_idx, h_idx, Chem.rdchem.BondType.SINGLE)

    Chem.SanitizeMol(mol)
    return mol


### Exercise: Round-trip accuracy (RDKit ⇄ NetworkX)

The goal of this exercise is to verify that converting

RDKit → NetworkX → RDKit

preserves the **chemical information we care about**.

You should treat the two functions provided above as a black box.

---

#### Q1 — Heavy-atom SMILES invariance

Write a function `roundtrip_smiles_equal(smiles)` that:

1. parses a SMILES string into an RDKit molecule,
2. converts it to a typed graph using `mol_to_graph`,
3. reconstructs a molecule using `graph_to_mol`,
4. compares the **canonical heavy-atom SMILES** of the original and reconstructed molecules.

The function should return `True` if the two SMILES are identical, and `False` otherwise.

---

#### Q2 — Count invariants

Extend your check in **Q1** to also verify that:

- the number of **heavy atoms** is preserved,
- the number of **heavy-atom bonds** is preserved.

Return `True` only if *all* invariants are satisfied.

> Hint: use `Chem.RemoveHs(mol)` before counting atoms or bonds.

---


<details>
<summary><b>Solution:</b></summary>

### Q1–Q2: Round-trip checker (heavy SMILES + count invariants)

```python
from rdkit import Chem
from rdkit.Chem import rdmolops  # optional: useful for extra invariants

def canonical_heavy_smiles(m: Chem.Mol) -> str:
    """Return canonical SMILES after removing H (heavy-atom skeleton)."""
    return Chem.MolToSmiles(Chem.RemoveHs(m), canonical=True)

def heavy_counts(m: Chem.Mol) -> tuple[int, int]:
    """Return (n_heavy_atoms, n_heavy_bonds) after removing H."""
    mh = Chem.RemoveHs(m)
    return mh.GetNumAtoms(), mh.GetNumBonds()

def roundtrip_ok(smiles: str, verbose: bool = True) -> bool:
    """
    RDKit → NetworkX → RDKit round-trip check.

    Criteria:
    1) canonical heavy-atom SMILES preserved
    2) heavy atom count preserved
    3) heavy bond count preserved
    """
    m1 = Chem.MolFromSmiles(smiles)
    if m1 is None:
        if verbose:
            print("Parse failed:", smiles)
        return False

    G = mol_to_graph(m1, include_implicit_h=True)
    m2 = graph_to_mol(G, make_explicit_h=False)

    s1, s2 = canonical_heavy_smiles(m1), canonical_heavy_smiles(m2)
    c1, c2 = heavy_counts(m1), heavy_counts(m2)

    ok = (s1 == s2) and (c1 == c2)

    if verbose and not ok:
        print("FAIL:", smiles)
        print(" heavy SMILES:", s1, "vs", s2)
        print(" counts:", c1, "vs", c2)

    return ok
```

### Quick test (run on a small subset)

```python
n_test = min(50, len(df))
fails = []

for s in df["smiles"].head(n_test):
    if not roundtrip_ok(s, verbose=False):
        fails.append(s)

print("Checked:", n_test)
print("Failures:", len(fails))
if fails:
    print("Example failures:", fails[:5])

# Optional: inspect one failure in detail
if fails:
    _ = roundtrip_ok(fails[0], verbose=True)
```


## 3. Isomorphism

We connect **formal graph-morphism definitions** to concrete
`networkx` matchers used in practice.

### Attribute predicates

We denote the **vertex** and **edge** attribute predicates as:

$$
\Phi_V : V(G) \times V(H) \rightarrow \{\text{true}, \text{false}\}
$$

$$
\Phi_E : E(G) \times E(H) \rightarrow \{\text{true}, \text{false}\}
$$

In code, these predicates are implemented as Python functions:

- `node_match(n1, n2)` — compares atom (vertex) attributes
- `edge_match(e1, e2)` — compares bond (edge) attributes

### Compatibility used in S01

For **S01**, we adopt a *strict-but-minimal* chemical compatibility model:

- same atom `symbol` (element),
- same `formal_charge`,
- same `aromatic` flag,
- same bond `order`.


In [6]:
from rdkit import Chem
from networkx.algorithms import isomorphism as iso

pairs = {
    "benzene": ("c1ccccc1", "C1=CC=CC=C1"),
    "aniline": ("c1ccccc1N", "c1ccccc1[NH3+]"),
}

graphs = {}
for name, (sa, sb) in pairs.items():
    graphs[f"{name}_a"] = mol_to_graph(Chem.MolFromSmiles(sa))
    graphs[f"{name}_b"] = mol_to_graph(Chem.MolFromSmiles(sb))


def node_match(n1, n2):
    return n1.get("symbol") == n2.get("symbol")

def edge_match(e1, e2):
    return int(e1.get("order", 1)) == int(e2.get("order", 1))

def iso_and_count(G1, G2, nm, em):
    gm = iso.GraphMatcher(G1, G2, node_match=nm, edge_match=em)
    return gm.is_isomorphic(), sum(1 for _ in gm.isomorphisms_iter())

print("=== simple matcher (symbol + order) ===")
for name in pairs:
    G1 = graphs[f"{name}_a"]; G2 = graphs[f"{name}_b"]
    iso_flag, n_maps = iso_and_count(G1, G2, node_match, edge_match)
    print(f"{name:8} | isomorphic: {int(iso_flag):1d} | mappings: {n_maps}")


=== simple matcher (symbol + order) ===
benzene  | isomorphic: 1 | mappings: 12
aniline  | isomorphic: 1 | mappings: 2


### Exercise: Isomorphism
**Q3 — Fix the matcher**

Implement `node_match` that requires matching `symbol` **and** either `total_h` or `formal_charge` (or both). Replace the existing `node_match` with your function and re-run the demo so that:

- `benzene` still matches, and  
- `aniline` (`c1ccccc1N`) **does not** match `anilinium` (`c1ccccc1[NH3+]`).

> Hint: `mol_to_graph(..., include_implicit_h=True)` stores H as `total_h`. Use `n.get("total_h",0)` or `n.get("formal_charge",0)`.



<details> <summary><b>Solution:</b></summary>

```python
# Solution: enhanced matcher that checks symbol + (total_h OR formal_charge)
def enhanced_node_match(n1, n2):
    return (
        n1.get("symbol") == n2.get("symbol")
        and (
            int(n1.get("total_h", 0)) == int(n2.get("total_h", 0))
            or int(n1.get("formal_charge", 0)) == int(n2.get("formal_charge", 0))
        )
    )

# run the demo with the enhanced matcher (uses existing `pairs`, `graphs`, `edge_match`, `iso_and_count`)
print("=== enhanced matcher (symbol + total_h/charge) ===")
for name in pairs:
    G1 = graphs[f"{name}_a"]; G2 = graphs[f"{name}_b"]
    iso_flag, n_maps = iso_and_count(G1, G2, enhanced_node_match, edge_match)
    print(f"{name:8} | isomorphic: {int(iso_flag):1d} | mappings: {n_maps}")

# quick instructor checks
assert iso_and_count(graphs["benzene_a"], graphs["benzene_b"], enhanced_node_match, edge_match)[0]
assert not iso_and_count(graphs["aniline_a"], graphs["aniline_b"], enhanced_node_match, edge_match)[0]
```
<details>

## 4. Automorphisms & orbits

**Observation.** In the benzene example you enumerated **12 mappings** — these are the automorphisms of the benzene heavy-atom graph (the dihedral group \(D_6\), where \(|D_6| = 12\)).

**Definition.** An automorphism is a graph isomorphism from the graph to itself:

$$
f : G \longrightarrow G.
$$

The automorphism group is

$$
\mathrm{Aut}(G).
$$

The **orbit** of a vertex \(v\) is the set of images of \(v\) under all automorphisms:

$$
\mathrm{Orbit}(v)=\{\psi(v)\;|\;\psi\in\mathrm{Aut}(G)\}.
$$

**Facts.** For benzene:

$$
|\mathrm{Aut}(G)| = |D_6| = 12,
$$

and all six carbon atoms lie in a single orbit.

**Why it matters.** Symmetric hosts produce many equivalent embeddings → duplicate matches and wasted work.

**Simple remedies.**
- Deduplicate by host-atom set: use `frozenset(mapping.values())`.  
- Use orbit representatives (e.g. choose the $\min$ index per orbit).  
- Accept only a canonical mapping (WL/lexicographic tie-break).

**Practical tips.**
- Include chemical attributes (`total_h`, `formal_charge`, stereochemistry) in matchers to reduce false symmetry.  
- Pre-filter with cheap signatures (degree, label counts, WL hashes) before enumerating automorphisms.



### Exercise: Automorphisms of a Molecular Graph

### Q4 — Develop a function `enumerate_automorphisms` to enumerate all automorphisms of a graph

**Hint:** An automorphism of a graph \(G\) is an isomorphism from \(G\) to itself.

Equivalently, the automorphism group satisfies

$$
\mathrm{Aut}(G) \subseteq \mathrm{Iso}(G, G).
$$



<details> <summary><b>Solution:</b></summary>

```python
def enumerate_automorphisms(G: nx.Graph):
    GM_self = iso.GraphMatcher(G, G, node_match=node_match, edge_match=edge_match)
    return list(GM_self.isomorphisms_iter())
```

In [ ]:
# Instructor utility (used later): enumerate automorphisms via NetworkX GraphMatcher
# You can treat this as the computational counterpart of "Aut(G)" in the theory section.

from networkx.algorithms import isomorphism as iso

def enumerate_automorphisms(G: nx.Graph):
    GM_self = iso.GraphMatcher(G, G, node_match=node_match, edge_match=edge_match)
    return list(GM_self.isomorphisms_iter())


We can now analyse the symmetry of a molecular graph by computing the
**orbits induced by its automorphism group**.

Under the natural action of the automorphism group on the vertex set,
two vertices belong to the same orbit if there exists an automorphism
mapping one to the other.

$$
\text{For } u, v \in V(G), \quad
u \sim v
\;\Longleftrightarrow\;
\exists\, \varphi \in \mathrm{Aut}(G)
\text{ such that }
\varphi(u) = v .
$$

Each orbit therefore represents a set of **symmetry-equivalent atoms**.


In [44]:
import networkx as nx
from typing import Dict, Iterable, List, Set


def compute_orbits_from_automorphisms(
    G: nx.Graph,
    automorphisms: Iterable[Dict] | None = None,
) -> List[Set]:
    """
    Compute vertex orbits induced by the automorphism group of a graph.

    Given the automorphism group Aut(G) acting on V(G), two vertices
    u, v ∈ V(G) belong to the same orbit if there exists an automorphism
    φ ∈ Aut(G) such that φ(u) = v.

    This function computes the orbits by collapsing vertices connected
    by automorphism mappings using a union–find (disjoint-set) structure.

    Parameters
    ----------
    G : nx.Graph
        Input graph.
    automorphisms : iterable of dict, optional
        Precomputed automorphisms φ : V(G) → V(G).
        If None, they are computed internally.

    Returns
    -------
    List[Set]
        List of vertex orbits. Each orbit is a set of nodes.
        Ordering is deterministic (sorted by smallest element).
    """
    if automorphisms is None:
        automorphisms = enumerate_automorphisms(G)

    # --- Disjoint-set (union–find) structure ---
    parent: Dict = {v: v for v in G.nodes()}

    def find(x):
        """Find representative with path compression."""
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]

    def union(a, b):
        """Union sets containing a and b."""
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    # --- Apply group action ---
    for auto in automorphisms:
        for v, fv in auto.items():
            union(v, fv)

    # --- Collect orbits ---
    orbits: Dict = {}
    for v in G.nodes():
        r = find(v)
        orbits.setdefault(r, set()).add(v)

    # deterministic ordering (useful for teaching & testing)
    return sorted(orbits.values(), key=lambda s: min(s))

from rdkit import Chem

benzene = Chem.MolFromSmiles("c1ccccc1")
G_bz = mol_to_graph(benzene)

autos = enumerate_automorphisms(G_bz)
orbits = compute_orbits_from_automorphisms(G_bz, autos)

print("Number of automorphisms (benzene):", len(autos))
print("Orbits:", orbits)


Number of automorphisms (benzene): 12
Orbits: [{0, 1, 2, 3, 4, 5}]


---

## Next: S02

In **S02** we focus on **subgraph isomorphism** (pattern → host), why symmetry explodes the number of matches,
and how **MCS** relates to atom-mapping and alignment.